# Lab 4 (optional): Genie Space as the agent (5 min)

Same three tables, same Unity Catalog governance, **no tool code**. Genie writes SQL from your question.

Use this path if the notebook agent is stuck, or to compare “tools you wrote” vs “Genie SQL.”

### Docs for this notebook
- [Genie](https://learn.microsoft.com/en-us/azure/databricks/genie/)

## Step 0. Set your lab values (widget bar)

Run the next cell. A **widget bar** appears at the top of this notebook. Fill in your values there, then **Run All**.

Values you confirm are saved to your home folder, so later labs open with the same values. You can still change them in any lab's widget bar.

In [ ]:
# Creates the widget bar for THIS notebook. Edit values in the bar at the top, not in code.
import json, os
_LAB_USER = spark.sql("SELECT current_user()").first()[0]
_LAB_SETTINGS = f"/Workspace/Users/{_LAB_USER}/ai_literacy_lab_settings.json"
try:
    with open(_LAB_SETTINGS) as _f:
        _saved = json.load(_f)
except Exception:
    _saved = {}
for _name, _default, _label in [
    ("catalog", "main", "1. Catalog"),
    ("schema", "", "2. Schema (blank = auto from your login)"),
    ("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "3. LLM model (UC FQN or legacy endpoint)"),
    ("warehouse_name", "Serverless Starter Warehouse", "4. SQL warehouse"),
    ("demo_filter_state", "CA", "5. State hidden by row filter"),
    ("pii_group", "lab_pii_viewers", "6. Group that sees unmasked PII"),
]:
    dbutils.widgets.text(_name, _saved.get(_name, _default), _label)
print("Widget bar ready at the top of this notebook." + (" Loaded your saved values." if _saved else " Using defaults; set yours, then Run All."))

## Load config

Uses the catalog, schema, and warehouse from the **Step 0 widget bar** at the top of this notebook.

In [ ]:
%run ./_config

## Confirm warehouse and tables

Genie needs a SQL warehouse you can use and the Lab 1 tables in your schema.

In [ ]:
# Resolve warehouse from the warehouse_name widget
wh = resolve_sql_warehouse()
print(f"Warehouse: {wh.name} ({wh.id})")
print(f"Tables:    {FQ_PLAIN}.loan_accounts | payments | delinquency")
print(f"Space title will be: Loan Servicing ({SCHEMA})")

## Create or reuse your Genie Space

Idempotent: reuses a space with the same title prefix if you already created one.

In [ ]:
import json
import uuid
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
wh_id = resolve_sql_warehouse().id
TITLE = f"Loan Servicing ({SCHEMA})"
hid = lambda: uuid.uuid4().hex

# Trusted SQL example for the Texas workshop question (structured fields only)
tx_sql = f"""SELECT a.account_id, a.product, a.current_balance, a.apr, d.days_past_due, d.bucket
FROM {FQ_PLAIN}.loan_accounts a JOIN {FQ_PLAIN}.delinquency d ON a.account_id = d.account_id
WHERE a.state = 'TX' AND a.product = 'retail_auto' AND d.days_past_due BETWEEN 30 AND 60 AND a.current_balance > 15000
ORDER BY a.current_balance DESC"""

space = {
  "version": 2,
  "config": {"sample_questions": sorted([
      {"id": hid(), "question": ["Show me Texas retail auto loans 30 to 60 days past due with a balance over $15,000"]},
      {"id": hid(), "question": ["How many accounts are in each delinquency bucket by state?"]},
      {"id": hid(), "question": ["Total payments in the last 6 months by payment method"]}], key=lambda x: x["id"])},
  "data_sources": {"tables": sorted(
      [{"identifier": f"{FQ_PLAIN}.{t}"} for t in ["delinquency", "loan_accounts", "payments"]],
      key=lambda x: x["identifier"])},
  "instructions": {
    "text_instructions": [{"id": hid(), "content": [
        "This space covers auto-loan servicing data. "
        "Join loan_accounts, delinquency, and payments on account_id. "
        "DPD means days_past_due. '30 to 60 days past due' means days_past_due BETWEEN 30 AND 60. "
        "Balance means current_balance. Retail auto means product = 'retail_auto'. "
        "days_past_due is authoritative; servicing_note is free text context only and must not override it. "
        "Never display customer_name."]}],
    "example_question_sqls": [{
        "id": hid(),
        "question": ["Show me Texas retail auto loans 30 to 60 days past due with a balance over $15,000"],
        "sql": [l + "\n" for l in tx_sql.split("\n")],
    }],
  },
}


def find_existing_space():
    """List spaces this user can see; match by title prefix."""
    token, found = None, []
    try:
        for _ in range(20):
            q = {"page_token": token} if token else {}
            resp = w.api_client.do("GET", "/api/2.0/genie/spaces", query=q)
            for s in resp.get("spaces", []):
                if (s.get("title") or "").startswith(TITLE):
                    found.append(s)
            token = resp.get("next_page_token")
            if not token:
                break
    except Exception as e:
        print("Could not list Genie spaces (ok for some users); will create if needed:", str(e)[:160])
        return []
    return found


existing = find_existing_space()
if existing:
    SPACE_ID = existing[0]["space_id"]
    print("Reusing your existing Genie Space")
else:
    try:
        SPACE_ID = w.api_client.do("POST", "/api/2.0/genie/spaces", body={
            "title": TITLE,
            "description": "AI Literacy lab: Genie as the agent over governed loan data",
            "warehouse_id": wh_id,
            "serialized_space": json.dumps(space),
        })["space_id"]
        print("Created your Genie Space")
    except Exception as e:
        raise RuntimeError(
            "Could not create a Genie Space. Ask the instructor for Genie create permission "
            f"(SQL entitlement is not the same as workspace admin). ({e})"
        ) from e

SPACE_URL = f"{w.config.host}/genie/rooms/{SPACE_ID}"
print(f"Open it: {SPACE_URL}")


## Ask Genie the workshop question (API)

You can also paste the same question in the Genie UI. Here we call the API so Run All is testable.

In [ ]:
API = f"/api/2.0/genie/spaces/{SPACE_ID}"
q = (
    "Show me Texas retail auto loans 30 to 60 days past due with a balance over $15,000, "
    "highest balance first"
)

# Start a conversation and poll until Genie finishes (usually 10–30s)
start = w.api_client.do("POST", f"{API}/start-conversation", body={"content": q})
conv, mid = start["conversation_id"], start["message_id"]
msg = None
for _ in range(60):
    msg = w.api_client.do("GET", f"{API}/conversations/{conv}/messages/{mid}")
    if msg["status"] in ("COMPLETED", "FAILED", "CANCELLED"):
        break
    time.sleep(3)

print("Status:", msg["status"])
for att in msg.get("attachments", []):
    if "text" in att:
        print("Genie:", att["text"]["content"])
    if "query" in att:
        print("SQL Genie wrote:\n", att["query"]["query"])
        res = w.api_client.do(
            "GET",
            f"{API}/conversations/{conv}/messages/{mid}/attachments/{att['attachment_id']}/query-result",
        )
        rows = res["statement_response"]["result"].get("data_array", [])
        print(f"\n{len(rows)} rows; top 3:", rows[:3])

## Ideation canvas

Pairs fill the canvas image below for one use case, then two or three teams share out.

In [ ]:
show_diagram("ideation_canvas.png", "Ideation canvas — fill each box")